In [ ]:
import sys
print(sys.executable)

In [ ]:
import os
os.chdir("/root/Ligand_Prep_trial")

Evaluate all minimized ligands against each ER protein

In [ ]:
import subprocess
import pandas as pd
import os

# =========================
# CONFIG
# =========================
GNINA = "~/gnina"  # remove the !, subprocess handles execution
MODE = 1  # use pose 1 consistently

# Folders
protein_dir = "proteinprep"
ligand_dir = "ligandprep"
ideal_ligand_dir = "minimized_ideal_ligand"  # NEW folder for ideal ligands
output_dir = "docked_results_min"

pairs = [
    {"pdb": "1ERE_A_fixed.pdb", "ligand": "EST_redock_1ERE_A.sdf", "ideal": "EST_min.sdf", "lig_id": "EST"},
    {"pdb": "1G50_A_fixed.pdb", "ligand": "EST_redock_1G50_A.sdf", "ideal": "EST_min.sdf", "lig_id": "EST"},
    {"pdb": "1GWR_A_fixed.pdb", "ligand": "EST_redock_1GWR_A.sdf", "ideal": "EST_min.sdf", "lig_id": "EST"},
    {"pdb": "3ERD_A_fixed.pdb", "ligand": "DES_redock_3ERD_A.sdf", "ideal": "DES_min.sdf", "lig_id": "DES"},
    {"pdb": "3UU7_A_fixed.pdb", "ligand": "2OH_redock_3UU7_A.sdf", "ideal": "2OH_min.sdf", "lig_id": "2OH"},
    {"pdb": "3UUD_A_fixed.pdb", "ligand": "EST_redock_3UUD_A.sdf", "ideal": "EST_min.sdf", "lig_id": "EST"},
    {"pdb": "4MG8_A_fixed.pdb", "ligand": "27J_redock_4MG8_A.sdf", "ideal": "27J_min.sdf", "lig_id": "27J"},
    {"pdb": "4MG9_A_fixed.pdb", "ligand": "27K_redock_4MG9_A.sdf", "ideal": "27K_min.sdf", "lig_id": "27K"},
    {"pdb": "4MGA_A_fixed.pdb", "ligand": "27L_redock_4MGA_A.sdf", "ideal": "27L_min.sdf", "lig_id": "27L"},
    {"pdb": "4MGC_A_fixed.pdb", "ligand": "27M_redock_4MGC_A.sdf", "ideal": "27M_min.sdf", "lig_id": "27M"},
     {"pdb": "4TUZ_A_fixed.pdb", "ligand": "36J_redock_4TUZ_A.sdf", "ideal": "36J_min.sdf", "lig_id": "36J"},
     {"pdb": "4ZN7_A_fixed.pdb", "ligand": "DES_redock_4ZN7_A.sdf", "ideal": "DES_min.sdf", "lig_id": "DES"},
     {"pdb": "6CBZ_A_fixed.pdb", "ligand": "EST_redock_6CBZ_A.sdf", "ideal": "EST_min.sdf", "lig_id": "EST"},
]

# =========================
# HELPERS
# =========================
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:\n", cmd)
        print(result.stderr)
        raise RuntimeError("Execution stopped")
    return result.stdout


def parse_cnn_from_table(output, mode=1):
    for line in output.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(str(mode)):
            parts = [p for p in line.split() if p.replace('.', '', 1).replace('-', '', 1).isdigit()]
            if len(parts) < 3:
                continue
            vina_affinity = float(parts[1])
            cnn_pose = float(parts[3])
            cnn_affinity = float(parts[4])
            cnn_vs = cnn_pose * cnn_affinity
            return vina_affinity, cnn_pose, cnn_affinity, cnn_vs
    return None, None, None, None


def get_rmsd(ref, sdf, mode=1):
    out = run_cmd(f"obrms -f {ref} {sdf}")
    rmsds = [float(l.split()[-1]) for l in out.splitlines() if l.strip().startswith("RMSD")]
    return rmsds[mode - 1] if len(rmsds) >= mode else None

# =========================
# DOCKING LOOP (ALL IDEALS)
# =========================
results = []

# Get all ideal ligands in folder
all_ideal_sdfs = [f for f in os.listdir(ideal_ligand_dir) if f.endswith(".sdf")]

for p in pairs:
    lig = p["lig_id"]
    print(f"\n🚀 Docking into {p['pdb']}")

    receptor_file = os.path.join(protein_dir, p['pdb'])
    native_ligand_file = os.path.join(ligand_dir, p['ligand'])

    for ideal_name in all_ideal_sdfs:

        ideal_file = os.path.join(ideal_ligand_dir, ideal_name)
        ideal_id = os.path.splitext(ideal_name)[0]

        print(f"   → Ideal ligand: {ideal_name}")

        ideal_out = os.path.join(output_dir, f"docked_{p['pdb']}_{ideal_id}.sdf")

        out = run_cmd(
            f"{GNINA} -r {receptor_file} "
            f"-l {ideal_file} "
            f"--autobox_ligand {native_ligand_file} "
            f"--seed 0 --exhaustiveness 16 "
            f"-o {ideal_out}"
        )

        vina, pose, aff, vs = parse_cnn_from_table(out, MODE)

        # =========================
        # RMSD ONLY FOR EXPLICIT PAIR
        # =========================
        if ideal_name == p["ideal"]:
            rmsd = get_rmsd(native_ligand_file, ideal_out, MODE)
        else:
            rmsd = None

        results.append({
            "ideal_ligand": ideal_id,
            "protein": p["pdb"],
            "ligand_native": lig,
            "dock_type": "ideal_minimized",
            "vina_affinity": vina,
            "CNNpose": pose,
            "CNNaffinity": aff,
            "CNN_VS": vs,
            "RMSD": rmsd
        })

# =========================
# RESULTS
# =========================
df_min = pd.DataFrame(results)

# Sort by ideal_ligand (alphabetical)
df_min = df_min.sort_values(by="ideal_ligand").reset_index(drop=True)

df_min

In [ ]:
# =========================
# Add Ligand Name and Ligand Type
# =========================

# Create mapping dictionaries
ligand_name_map = {
    "27J_min": "alpha-zearalanol",
    "27K_min": "Butyl paraben",
    "27L_min": "4-tert-octylphenol",
    "27M_min": "Benzophenone-2",
    "2OH_min": "Bisphenol A (BPA)",
    "36J_min": "alpha-zearalenol",
    "Caffeine_min": "Caffeine",
    "DES_min": "Diethylstilbestrol (DES)",
    "EE2_min": "EE2",
    "EST_min": "17β-estradiol (E2)",
    "Melatonin_min": "Melatonin",
    "Testosterone_min": "Testosterone",
}

ligand_type_map = {
    "27J_min": "Moderate Antagonist",
    "27K_min": "Weak agonist/Strong Antagonist",
    "27L_min": "Weak agonist",
    "27M_min": "Agonist",
    "2OH_min": "Weak Agonist/Weak Antagonist",
    "36J_min": "Antagonist",
    "Caffeine_min": "Inactive",
    "DES_min": "Agonist",
    "EE2_min": "Agonist",
    "EST_min": "Agonist",
    "Melatonin_min": "Inactive",
    "Testosterone_min": "Weak agonist/moderate antagonist",
}

# Map the columns
df_min["Ligand Name"] = df_min["ideal_ligand"].map(ligand_name_map)
df_min["Ligand Type"] = df_min["ideal_ligand"].map(ligand_type_map)

# Optional: sort by ideal_ligand again
df_min = df_min.sort_values(by="ideal_ligand").reset_index(drop=True)

# Show the updated dataframe
df_min.head(20)

In [ ]:
df_min.to_excel("results/docking_results_ER_Minimized Ligand.xlsx", index=False)

In [ ]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Metadata
# -----------------------------
now = datetime.now()
report_time = now.strftime("%Y-%m-%d %H:%M:%S")
experiment_id = int(now.timestamp())

# -----------------------------
# Output directory
# -----------------------------
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

outfile = results_dir / f"gnina_docking_report_{experiment_id}.txt"

# -----------------------------
# Maps for ligand names and types
# -----------------------------
ligand_name_map = {
    "27J_min": "alpha-zearalanol",
    "27K_min": "Butyl paraben",
    "27L_min": "4-tert-octylphenol",
    "27M_min": "Benzophenone-2",
    "2OH_min": "Bisphenol A (BPA)",
    "36J_min": "alpha-zearalenol",
    "Caffeine_min": "Caffeine",
    "DES_min": "Diethylstilbestrol (DES)",
    "EE2_min": "EE2",
    "EST_min": "17β-estradiol (E2)",
    "Melatonin_min": "Melatonin",
    "Testosterone_min": "Testosterone",
}

ligand_type_map = {
    "27J_min": "Moderate Antagonist",
    "27K_min": "Weak agonist/Strong Antagonist",
    "27L_min": "Weak agonist",
    "27M_min": "Agonist",
    "2OH_min": "Weak Agonist/Weak Antagonist",
    "36J_min": "Antagonist",
    "Caffeine_min": "Inactive",
    "DES_min": "Agonist",
    "EE2_min": "Agonist",
    "EST_min": "Agonist",
    "Melatonin_min": "Inactive",
    "Testosterone_min": "Weak agonist/moderate antagonist",
}

# -----------------------------
# Header
# -----------------------------
header = f"""Report produced {report_time} | experiment id: {experiment_id}

GNINA cross-docking run:
Each estrogen receptor protein was docked against all minimized ideal ligands.
RMSD is reported only for matching native ligand pairs.

Ligands included in this run:
Ligand ID        Ligand Name                    Ligand Type
"""

# Insert ligand summary immediately after the line above
for lid in sorted(df_min['ideal_ligand'].unique()):
    name = ligand_name_map.get(lid, "NA")
    ltype = ligand_type_map.get(lid, "NA")
    header += f"{lid:<15} {name:<30} {ltype}\n"

# Add GNINA options and results table header
header += "\nGnina options:\nexhaustiveness 16\nseed 0\n\nResults:\nprotein + native_ligand | docked_ligand        CNN Score     CNN_VS       RMSD\n"

# -----------------------------
# Build results rows
# -----------------------------
rows = []
for _, r in df_min.iterrows():
    label = f"{r['protein']} + {r['ligand_native']} | {r['ideal_ligand']}"
    cnn_score = r["CNNpose"]
    cnn_vs = r["CNN_VS"]

    if r["RMSD"] is None:
        rmsd = "NA"
    elif r["RMSD"] == float("inf"):
        rmsd = "inf"
    else:
        rmsd = f"{r['RMSD']:.6f}"

    rows.append(f"{label:<60} {cnn_score:>10.4f} {cnn_vs:>12.6f} {rmsd:>12}")

# -----------------------------
# Write report
# -----------------------------
report = header + "\n".join(rows)

with open(outfile, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"\nSaved report to: {outfile.resolve()}")